In [1]:
# sensitivity_M2_grid_cost.py
"""
M2: Grid-connection cost sensitivity analysis.
Addresses reviewer concerns about MISO cost proxy and straight-line distance assumption.

Two dimensions:
  (a) Terrain detour multiplier κ  {1.0, 1.3, 1.6}  — effective routing distance
  (b) Cost multiplier              {0.5, 1.0, 2.0}   — uncertainty in construction cost

Key insight: since grid_capex = distance × cost_per_km × capacity_MW,
varying κ is equivalent to varying the effective cost_per_km.
Effective cost = κ × cost_multiplier × baseline_cost_per_km
"""

import pandas as pd
import numpy as np
from itertools import product
import yaml
from scipy.stats import spearmanr

# ── 1. Load configuration ─────────────────────────────────────────
with open("config/config_CAN_baseline.yaml", "r") as f:  # adjust to your BC config
    config = yaml.safe_load(f)

BASELINE_COST_PER_KM = config["transmission"]["grid_connection_cost_per_Km"]  # M$/km
TX_REBUILD_COST      = config["transmission"]["tx_line_rebuild_cost"]           # M$/km

# ── 2. Load pre-computed baseline cell results ────────────────────
# These come from your previous full run — no ERA5 re-download needed.
# Columns needed: cell_id, region, technology, cf_annual,
#                 nearest_node_km, max_capacity_MW, capex_usd_per_kw,
#                 fom_usd_per_kw_yr, vom_usd_per_mwh
df_base = pd.read_csv("results/BC_baseline_cell_scores.csv")  # adjust path

# ── 3. Import CellScorer ──────────────────────────────────────────
from RES.scorer import CellScorer   # adjust import path if different

# ── 4. Define sensitivity parameter grid ─────────────────────────
kappa_values    = [1.0, 1.3, 1.6]   # terrain detour multipliers
cost_multipliers = [0.5, 1.0, 2.0]  # cost uncertainty multipliers

# CRF parameters (from config or hardcoded baseline)
DISCOUNT_RATE = config.get("scorer", {}).get("discount_rate", 0.07)  # adjust key
PROJECT_LIFE  = config.get("scorer", {}).get("project_life",  25)
REFERENCE_CAPACITY_MW = 100   # as in manuscript

# ── 5. Run sensitivity loop ───────────────────────────────────────
records = []

for kappa, cost_mult in product(kappa_values, cost_multipliers):
    effective_cost_per_km = kappa * cost_mult * BASELINE_COST_PER_KM

    for _, row in df_base.iterrows():
        # Re-compute grid connection CAPEX with new effective cost
        spur_km = row["nearest_node_km"] * kappa  # terrain-adjusted distance
        grid_capex_musd = spur_km * effective_cost_per_km * row["max_capacity_MW"] / 1000.0
        # Note: grid_capex_musd is in M$; convert to $/kW for CellScorer
        grid_capex_usd_per_kw = (grid_capex_musd * 1e6) / (row["max_capacity_MW"] * 1e3)

        scorer = CellScorer(
            capacity_mw        = row["max_capacity_MW"],
            cf_annual           = row["cf_annual"],
            capex_usd_per_kw    = row["capex_usd_per_kw"] + grid_capex_usd_per_kw,
            fom_usd_per_kw_yr   = row["fom_usd_per_kw_yr"],
            vom_usd_per_mwh     = row["vom_usd_per_mwh"],
            discount_rate       = DISCOUNT_RATE,
            project_life_yrs    = PROJECT_LIFE,
        )
        lcoe_proxy = scorer.compute_score()   # returns CAD/MWh (adjust method name)

        records.append({
            "cell_id":       row["cell_id"],
            "region":        row["region"],
            "technology":    row["technology"],
            "kappa":         kappa,
            "cost_mult":     cost_mult,
            "effective_cost_per_km": effective_cost_per_km,
            "lcoe_proxy":    lcoe_proxy,
            "max_capacity_MW": row["max_capacity_MW"],
        })

df_results = pd.DataFrame(records)

# ── 6. Rank-order correlation (Spearman ρ) ───────────────────────
print("\n=== Spearman ρ of top-50 site rankings vs. baseline ===")
baseline_mask = (df_results["kappa"] == 1.0) & (df_results["cost_mult"] == 1.0)

for tech in ["wind", "solar"]:
    baseline_scores = (df_results[baseline_mask & (df_results["technology"] == tech)]
                       .set_index("cell_id")["lcoe_proxy"])
    top50_baseline = baseline_scores.nsmallest(50).index

    print(f"\n  {tech.upper()}")
    for kappa, cost_mult in product(kappa_values, cost_multipliers):
        if kappa == 1.0 and cost_mult == 1.0:
            continue
        mask = ((df_results["kappa"] == kappa) &
                (df_results["cost_mult"] == cost_mult) &
                (df_results["technology"] == tech))
        variant_scores = df_results[mask].set_index("cell_id")["lcoe_proxy"]
        # Align to same cells
        shared = baseline_scores.index.intersection(variant_scores.index)
        rho, _ = spearmanr(baseline_scores[shared], variant_scores[shared])
        print(f"    κ={kappa:.1f}, cost×{cost_mult:.1f} → ρ = {rho:.3f}")

# ── 7. Supply curve comparison ────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, tech in zip(axes, ["wind", "solar"]):
    tech_df = df_results[df_results["technology"] == tech]

    for (kappa, cost_mult), grp in tech_df.groupby(["kappa", "cost_mult"]):
        grp_sorted = grp.sort_values("lcoe_proxy")
        cum_cap = grp_sorted["max_capacity_MW"].cumsum() / 1000  # GW
        lcoe    = grp_sorted["lcoe_proxy"]
        lw = 2.5 if (kappa == 1.0 and cost_mult == 1.0) else 1.0
        ls = "-" if cost_mult == 1.0 else ("--" if cost_mult == 0.5 else ":")
        label = f"κ={kappa:.1f}, ×{cost_mult:.1f}"
        ax.step(cum_cap, lcoe, where="post", linewidth=lw, linestyle=ls, label=label)

    ax.set_xlabel("Cumulative Capacity (GW)")
    ax.set_ylabel("Screening-level LCOE proxy (CAD/MWh)")
    ax.set_title(f"{tech.upper()} — Grid Cost Sensitivity")
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results/M2_grid_cost_sensitivity_supply_curves.png", dpi=200, bbox_inches="tight")

# ── 8. Aggregate sensitivity table ───────────────────────────────
agg = (df_results
       .groupby(["technology", "kappa", "cost_mult"])
       .agg(
           total_cap_GW   = ("max_capacity_MW", lambda x: x.sum() / 1000),
           median_lcoe    = ("lcoe_proxy", "median"),
           p25_lcoe       = ("lcoe_proxy", lambda x: x.quantile(0.25)),
           p75_lcoe       = ("lcoe_proxy", lambda x: x.quantile(0.75)),
       )
       .reset_index())
agg.to_csv("results/M2_grid_sensitivity_table.csv", index=False)
print("\n=== M2 sensitivity table saved ===")
print(agg.to_string())

KeyError: 'transmission'